In [ ]:
%pip install sammo --upgrade

### Calling an LLM

Sammo makes it easy to call an LLM and present the results in a structured way.

In [8]:
import getpass
import os
from pathlib import Path
from sammo.components import GenerateText, Output
from sammo.runners import OpenAIChat

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

BASE_DIR = Path.cwd()
if not (BASE_DIR / "transaction_data_with_classifications.csv").exists():
    BASE_DIR = BASE_DIR / "prompt_optimization_and_evals"

MODEL = os.getenv("SAMMO_MODEL", "gpt-4o")
# SAMMO 0.3.3 sends max_tokens=None when this is omitted.
MAX_OUTPUT_TOKENS = int(os.getenv("SAMMO_MAX_OUTPUT_TOKENS", "1000"))
CACHE_FILE = str(BASE_DIR / os.getenv("CACHE_FILE", "cache.tsv"))

runner = OpenAIChat(
    model_id=MODEL,
    api_config={"api_key": os.environ["OPENAI_API_KEY"]},
    cache=CACHE_FILE,
    timeout=30,
    max_context_window=MAX_OUTPUT_TOKENS,
)

first = GenerateText(
    "Hello! My name is Mike and I work as a prompt engineer.",
    system_prompt="Talk like Shakespeare.",
)
Output(first).run(runner)


+---------+-------------------------------------------------------------+
| input   | output                                                      |
+=========+=============================================================+
| None    | Greetings, good sir Mike! 'Tis a pleasure to make thine     |
|         | acquaintance. Thou art a prompt engineer, say'st thou? A    |
|         | noble pursuit indeed, crafting and shaping the very essence |
|         | of inquiry. Pray, tell me more of thy noble endeavors in    |
|         | this field!                                                 |
+---------+-------------------------------------------------------------+
Constants: None

In [9]:
second = GenerateText("Write a four line poem about my job.", history=first)
poem = Output(second).run(runner)
poem

+---------+--------------------------------------------------------------+
| input   | output                                                       |
+=========+==============================================================+
| None    | In realms where words and thoughts entwine,   Doth Mike, the |
|         | prompt engineer, design.   With skillful craft, he doth      |
|         | inspire,   To kindle minds and set hearts afire.             |
+---------+--------------------------------------------------------------+
Constants: None

In [10]:
poem.outputs[0].plot_call_trace()

### Static Loops

You can loop over a static list of inputs and then combine them with Union.

In [11]:
from sammo.components import Union

N = 5
fruits = [
    GenerateText("Generate the name of an LLM model provided by OpenAI. Only respond with the name of the model.", randomness=0.9, seed=i)
    for i in range(N)
]
static_loop = Output(Union(*fruits))
static_loop.run(runner)

+---------+-----------------------------------------------+
| input   | output                                        |
+=========+===============================================+
| None    | ['GPT-4', 'GPT-4', 'GPT-4', 'GPT-4', 'GPT-4'] |
+---------+-----------------------------------------------+
Constants: None

In [12]:
static_loop.plot_program()

### Dynamic Loops

You can loop over all the results of the previous layer with ForEach.

In [13]:
from sammo.extractors import ExtractRegex
from sammo.components import ForEach
from sammo.base import Template

models = ExtractRegex(
    GenerateText(
        "Generate a list of 5 models by OpenAI. Wrap each model with <model> and </model>."
    ),
    r"<model>(.*?)<.?model>"
)

model_blurbs = Output(ForEach(
    "model",
    models,
    GenerateText(Template("Why is {{model}} a good model in less than 25 words?")),
))
model_desc = model_blurbs.run(runner)

In [14]:
model_desc.outputs[0].plot_call_trace()


In [15]:
model_blurbs.plot_program()


### Parallelization

Sammo automatically runs components in parallel where possible within a rate limit.

In [16]:
from sammo.data import DataTable

runner = OpenAIChat(
    model_id=MODEL,
    api_config={"api_key": os.environ["OPENAI_API_KEY"]},
    cache=CACHE_FILE,
    rate_limit=6, # default is 2 per second
    max_context_window=MAX_OUTPUT_TOKENS,
)
numbers = list(range(1,6))
spp = Output(GenerateText(Template("Output only the corresponding greek letter in English: {{input}}")))
spp.run(runner, DataTable(numbers))


minibatches[################################################################################]5/5[00:00<00:00, 4649.90it/s]


+---------+----------+
| input   | output   |
+=========+==========+
| 1       | Alpha    |
+---------+----------+
| 2       | Beta     |
+---------+----------+
| 3       | Gamma    |
+---------+----------+
| 4       | Delta    |
+---------+----------+
| 5       | Epsilon  |
+---------+----------+
Constants: None